In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder

from model import WeatherCNN
from model import WeatherLSTM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_EPOCH = 30
DATA_PATH = 'data/clean_data.csv'
SVG_MODEL_PATH = 'weather_model.pth'

## Prep

In [7]:
df = pd.read_csv(DATA_PATH) 

# data analysé
features_cols = [
    'City_Encoded', 'Month_sin', 'Month_cos', 'MinTemp', 'MaxTemp', 
    'Rainfall', 'WindGustSpeed', 'Humidity9am', 'Humidity3pm', 
    'Pressure9am', 'Pressure3pm', 'WindDir3pm_sin', 'WindDir3pm_cos'
]

# data attendu
target_temp = 'MaxTemp'
target_rain = 'RainTomorrow' # pourcentage

# normalisation des features
scaler = StandardScaler()
df[features_cols] = scaler.fit_transform(df[features_cols])

le_city = LabelEncoder()
le_city.fit(df['Location'])

LabelEncoder()

In [8]:
# Prépare des paquets de 7 jours de données pour prédire la météo du jour suivant, ville par ville
def create_sequences_by_city(df, seq_length):
    X, yt, yr = [], [], []
    for city, group in df.groupby('Location'):
        data = group[features_cols].values
        temp = group[target_temp].values
        rain = group[target_rain].values
        for i in range(len(data) - seq_length):
            X.append(data[i:i+seq_length])
            yt.append(temp[i+seq_length])
            yr.append(rain[i+seq_length])
    return np.array(X), np.array(yt), np.array(yr)

X_s, yt_s, yr_s = create_sequences_by_city(df, 7)

# Split
split = int(0.8 * len(X_s))
X_train = torch.tensor(X_s[:split], dtype=torch.float32).permute(0, 2, 1).to(device)
yt_train = torch.tensor(yt_s[:split], dtype=torch.float32).view(-1, 1).to(device)
yr_train = torch.tensor(yr_s[:split], dtype=torch.float32).view(-1, 1).to(device)

X_val = torch.tensor(X_s[split:], dtype=torch.float32).permute(0, 2, 1).to(device)
yt_val = torch.tensor(yt_s[split:], dtype=torch.float32).view(-1, 1).to(device)
yr_val = torch.tensor(yr_s[split:], dtype=torch.float32).view(-1, 1).to(device)

train_loader = DataLoader(TensorDataset(X_train, yt_train, yr_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, yt_val, yr_val), batch_size=64, shuffle=False)

# Calcul du poids pour le déséquilibre => Que la pluie ai autant d'importance que le pas pluie
num_neg = (yr_train == 0).sum()
num_pos = (yr_train == 1).sum()
pos_weight = (num_neg / num_pos).to(device)

# Entrainement

In [9]:
model = WeatherCNN(len(features_cols)).to(device)
# model = WeatherLSTM(len(features_cols), hidden_size=64, num_layers=2).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0005)
mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(N_EPOCH):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for b_x, b_yt, b_yr in train_loader:
        optimizer.zero_grad()
        p_temp, p_rain_logits = model(b_x)
        
        l_temp = mse_loss(p_temp, b_yt)
        l_rain = bce_loss(p_rain_logits, b_yr)
        loss = l_temp + l_rain
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Accuracy pluie
        preds = (torch.sigmoid(p_rain_logits) > 0.5).float()
        train_correct += (preds == b_yr).sum().item()
        train_total += b_yr.size(0)
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for bv_x, bv_yt, bv_yr in val_loader:
            pv_temp, pv_rain_logits = model(bv_x)
            v_loss = mse_loss(pv_temp, bv_yt) + bce_loss(pv_rain_logits, bv_yr)
            val_loss += v_loss.item()
            
            v_preds = (torch.sigmoid(pv_rain_logits) > 0.5).float()
            val_correct += (v_preds == bv_yr).sum().item()
            val_total += bv_yr.size(0)

    # Toutes les 5 époques
    if (epoch + 1) % 5 == 0 or epoch == 0:
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch {epoch+1:02d}/{N_EPOCH} | "
              f"Loss Train: {avg_train_loss:.4f} | Loss Val: {avg_val_loss:.4f} | "
              f"Acc Rain Train: {train_acc:.2f}% | Acc Rain Val: {val_acc:.2f}%")

Epoch 01/30 | Loss Train: 1.1823 | Loss Val: 1.1211 | Acc Rain Train: 62.71% | Acc Rain Val: 65.60%
Epoch 05/30 | Loss Train: 1.0331 | Loss Val: 1.0732 | Acc Rain Train: 68.42% | Acc Rain Val: 62.97%
Epoch 10/30 | Loss Train: 0.9224 | Loss Val: 1.1026 | Acc Rain Train: 73.33% | Acc Rain Val: 65.69%
Epoch 15/30 | Loss Train: 0.8216 | Loss Val: 1.2354 | Acc Rain Train: 77.31% | Acc Rain Val: 65.13%
Epoch 20/30 | Loss Train: 0.7374 | Loss Val: 1.3507 | Acc Rain Train: 80.74% | Acc Rain Val: 66.38%
Epoch 25/30 | Loss Train: 0.6679 | Loss Val: 1.5176 | Acc Rain Train: 83.36% | Acc Rain Val: 70.37%
Epoch 30/30 | Loss Train: 0.6179 | Loss Val: 1.7535 | Acc Rain Train: 85.31% | Acc Rain Val: 71.22%


In [10]:
torch.save({
    'model_state_dict': model.state_dict(),
    'city_encoder': le_city,
    'scaler': scaler,
    'features': features_cols,
    'pos_weight': pos_weight.item()
}, SVG_MODEL_PATH)